# 01 · The PyTorch training loop (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`). The cell after each gap checks it.
Solutions: `../solutions/guided/01_training_loop.md`.

**Traces to** `notes/ioai-task-patterns.md` P1, P3. Every T25/T26 baseline is this loop: `train()` /
`eval()`, `zero_grad → backward → step`, `.to(device)`, `torch.no_grad()` at inference (T25-RADAR,
T25-CHICKEN, T25-G-SPEECH). Checkpointing the best epoch is copied from T25-CHICKEN.

Runtime: ~1 minute on a T4 GPU, ~4 minutes on CPU.

In [ ]:
import copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 1. Data

`ToTensor()` turns a PIL image into a `float32` tensor of shape `(1, 28, 28)` in `[0, 1]`. The labels come back
as Python ints, and the default collate turns them into an `int64` tensor, which is what `CrossEntropyLoss` wants.

In [ ]:
tf = transforms.ToTensor()
train_full = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
test_set = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
train_set, val_set = Subset(train_full, range(0, 20000)), Subset(train_full, range(50000, 55000))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=512)
test_loader = DataLoader(test_set, batch_size=512)
xb, yb = next(iter(train_loader))
print(xb.shape, xb.dtype, yb.shape, yb.dtype)

## 2. Model

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                     # (B, 1, 28, 28) -> (B, 784)
            # TODO(g1): first Linear layer: how many inputs does a flattened 28x28 image have?
            nn.Linear(____, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 10),               # raw logits: NO softmax here
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

In [ ]:
assert model(xb.to(device)).shape == (128, 10)
print("g1 ok")

## 3. Loss

Logits `(B, C)` plus integer class targets `(B,)` call for the loss that applies `log_softmax` internally.

In [ ]:
# TODO(g2): the classification loss for raw logits + int64 class indices
loss_fn = ____

In [ ]:
logits = torch.tensor([[2.0, 0.0, -1.0]]); target = torch.tensor([0])
assert torch.isclose(loss_fn(logits, target), -F.log_softmax(logits, 1)[0, 0])
print("g2 ok")

## 4. One training epoch

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()                             # dropout ON
    total, n = 0.0, 0
    for xb, yb in loader:
        # TODO(g3): move the batch to the same device as the model
        xb, yb = ____
        logits = model(xb)
        loss = loss_fn(logits, yb)
        # TODO(g4): the three optimizer lines, in the right order
        ____
        total += loss.item() * len(xb); n += len(xb)
    return total / n

In [ ]:
probe = MLP().to(device); opt = torch.optim.SGD(probe.parameters(), lr=0.1)
l1 = train_one_epoch(probe, DataLoader(Subset(train_full, range(2000)), batch_size=64, shuffle=True), opt)
l2 = train_one_epoch(probe, DataLoader(Subset(train_full, range(2000)), batch_size=64, shuffle=True), opt)
assert l2 < l1, (l1, l2)
print(f"g3, g4 ok (loss {l1:.3f} -> {l2:.3f})")

## 5. Evaluation: eval mode, no autograd graph

In [ ]:
def predict(model, loader):
    # TODO(g5): switch dropout/batch-norm to inference behaviour
    ____
    outs = []
    # TODO(g6): don't build the autograd graph while predicting
    with ____:
        for xb, _ in loader:
            outs.append(model(xb.to(device)).cpu())
    return torch.cat(outs)

def accuracy(model, loader):
    logits = predict(model, loader)
    y = torch.cat([yb for _, yb in loader])
    # TODO(g7): fraction of rows whose highest logit is the true class (a Python float)
    return ____

In [ ]:
out = predict(model, val_loader)
assert not model.training, "predict must leave the model in eval mode"
assert not out.requires_grad, "predictions still carry an autograd graph"
acc0 = accuracy(model, val_loader)
assert isinstance(acc0, float) and 0.0 <= acc0 <= 1.0
print(f"g5, g6, g7 ok (untrained accuracy {acc0:.3f})")

## 6. Full fit: AdamW, cosine schedule, keep the best epoch

`model.state_dict()` returns **references** to the live tensors, so saving it without a copy means
"best_state" silently keeps changing as training continues.

In [ ]:
model = MLP().to(device)
EPOCHS = 5
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_acc, best_state = -1.0, None
for epoch in range(EPOCHS):
    t0 = time.time()
    tr_loss = train_one_epoch(model, train_loader, optimizer)
    scheduler.step()                      # once per epoch, AFTER the optimizer steps
    val_acc = accuracy(model, val_loader)
    if val_acc > best_acc:
        # TODO(g8): snapshot the weights so later epochs cannot overwrite them
        best_acc, best_state = val_acc, ____
    print(f"epoch {epoch}  loss {tr_loss:.3f}  val acc {val_acc:.4f}  lr {scheduler.get_last_lr()[0]:.2e}  {time.time() - t0:.1f}s")

In [ ]:
k = next(iter(best_state))
assert best_state[k].data_ptr() != model.state_dict()[k].data_ptr(), "best_state aliases the live weights"
model.load_state_dict(best_state)
test_acc = accuracy(model, test_loader)
assert test_acc > 0.84, test_acc
print(f"g8 ok, all gaps done (test accuracy of best epoch {test_acc:.4f})")

## 7. Loss / target cheat-sheet (no gaps, just run it)

| task | model output | target | loss |
|---|---|---|---|
| multi-class | `(B, C)` logits | `(B,)` **int64** | `CrossEntropyLoss` |
| per-pixel classes | `(B, C, H, W)` | `(B, H, W)` int64 | `CrossEntropyLoss` |
| binary / multi-label | `(B,)` or `(B, K)` logits | same shape, **float** | `BCEWithLogitsLoss` |
| regression | `(B,)` | `(B,)` float | `MSELoss` / `L1Loss` |

In [ ]:
bce = nn.BCEWithLogitsLoss()
logit = torch.randn(8, 1)                       # a model with one output unit
y_bin = torch.randint(0, 2, (8,))               # int labels 0/1
print(bce(logit.squeeze(1), y_bin.float()))     # squeeze to (8,) and cast to float
seg_logits, seg_target = torch.randn(2, 5, 4, 4), torch.randint(0, 5, (2, 4, 4))
print(nn.CrossEntropyLoss()(seg_logits, seg_target))